<hr/>

# [Generalized Linear Mixed Models](https://donnievin.github.io/ASDA_2/)
By: **Donovan Vincent Jr** - dvincen9@jh.edu <br/>
Adapted From: **Dr. Sergey Kushnarev's ASDA II** <br/>
Estimated Workthrough Time: **90 mins**

<hr/>

<h1><font color="blue">Introduction to GLMs</font></h1>

Topics: 
* Recalling Linear Models (LMs)
* Generalizing Linear Models to get GLMs
* Exponential Dispersion Family (EDFs)
* Linear Predictor, $\eta$
* Link functions, $g(\mu) = \eta$
* Expectation and Variances of EDFs
* Mean-Variance Relationship
* The likelihood equations

In [1]:
import time
import scipy
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go

from IPython.display import clear_output


# <font color='lightblue'> Recalling Linear Models </font>

> What is a linear model?

A Linear Model (LM) is a statistcal equation that fits `parameters` ($\beta_i$) to model a relationship between `predictor` variables ($X_i$) and `response` variables ($Y_i$), even when the data is noisy with `errors` ($\epsilon_i$). In general, these equations take the form:

<div align="center">

$$
\mathbf{Y} = \mathbf{X} \mathbf{\beta} + \mathbf{\epsilon}
$$

</div>


--- 


> Error Constraints

There are 3 requirements for the noise term: 
1. `Centered at 0`: $\mathbb{E}[\epsilon]=0$ <br>
    a. This ensures $\mathbb{E}[Y] = \mathbb{E}[X \beta + \epsilon] = \mathbb{E}[X \beta] + \mathbb{E}[\epsilon] = \mathbb{E}[X \beta]$

2. `Independent noise`: $Cov(\epsilon_i , \epsilon_j)=0$ <br>
    a. This follows the assumption that each observation is an independent random sample
    b. Required for Gauss-Markov Theorem to the Ordinary Least Squares (OLS) estimate Unbiased

3. `Homoscedasticity (constant variance)`: $Var(\epsilon_i)= \sigma^2$ <br>
    a. This ensures $Var[Y] = \mathbb{E}[X \beta + \epsilon] = \mathbb{E}[X \beta] + \mathbb{E}[\epsilon] = \mathbb{E}[X \beta]$\
    b. Required for Gauss-Markov Theorem to ensure tht the Ordinary Least Squares (OLS) estimate is the Best Linear Unbiased Estimator (BLUE)


--- 

> LMs for polynomials

When we call these LINEAR models, we are only referring to the powers of $\beta_i$. We are free to have anything we want in the design matrix, even powers. 

Example: 
* $y = \beta_0 + \beta_1 x_1 + \beta_2 x_1^2 + \beta_3 x_1^3 + ... + \beta_p x_1^p = \sum_{i=1}^p \beta_i x_1^i$
* $X = [1, x_1^2,  x_1^2 + x_1^3 + ... + x_1^p] \in \mathbb{R}^{nxp}$
* $y = X \beta$


In [8]:
#@markdown <font color='darkred'>Run to start tutorial. </font>

#@markdown Everytime this cell runs, new random data is generated but the E[Y] (being modeled) can capture them all

x1 = np.arange(0,10)
x2 = np.arange(10,20)

B0, B1, B2 = 4, 5, 6


while True:
    # In numpy: loc=E[Y] ; scale = Std[Y] ; size = number of samples
    y_ideal = (B2 * x2) + (B1 * x1) + B0
    y_data = (B2 * x2) + (B1 * x1) + B0 + np.random.normal(loc=0, scale=20, size=len(x1))

    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=x1, y=x2, z=y_ideal,
            mode='markers',
            name='Model: E[Y]'
        )
    )

    fig.add_trace(
        go.Scatter3d(
            x=x1, y=x2, z=y_data,
            mode='markers',
            name='Observed: XB + e'
        )
    )


    # Axis labels
    fig.update_layout(
        scene=dict(
            xaxis=dict(title='Predictor 1', range=[0, 10]),
            yaxis=dict(title='Predictor 2', range=[10, 20]),
            zaxis=dict(title='Response', range=[60,200]),
            camera=dict(
                eye=dict(x=-0.8, y=1.5, z=0.1)
            )
        ),
        title='LMs: Model the relationship between noisy variables'
    )

    clear_output(wait=True)
    fig.show()
    time.sleep(2)


KeyboardInterrupt: 

# Generalized Linear Models

[MORE EXAMPLES](https://en.wikipedia.org/wiki/Exponential_family#cite_note-7)


> 3 main components needed to generalize a GLM 

1. Random component: Each $y_i \sim$ i.i.d. EDF

2. Linear Predictor: $\eta = X\beta$

    a. X = nxp design matrix 

    b. $\beta$ = px1 parameter vector 

    c. $\eta$ = nx1 ??? 


3. Link function: 

# Exponential Dispersion Family (Random Component)

We can model the probability density function (PDF) / probability mass functions (PMF) of a variety of of distributions using a common form where Y can be sampled from an exponential dispersion family: $Y_i \sim EDF(\theta_i, \phi_i)$. Their probability functions can be written using the following form:


<div align="center">

$$
f(y_i | \theta_i, \phi_i) = exp[ \frac{y_i \theta_i - b(\theta_i)}{a(\phi)} + c(y_i, \theta_i)]
$$

</div>


where:
* $\theta_i$ = Natural Parameter
* $b(\theta_i)$ = cumulant function
* $\phi_i$ = Dispersion parameter
* $c(y_i, \theta_i)$ = ???

## Poisson

$y_i \sim Pois(\mu_i)$

$f(y_i|\mu_i) =  \frac{\mu_i^{y_i} e^{-\mu_i}}{y_i!}$  Note: Taylor series expansion $e^\mu = \sum_{y=1}^\infty \frac{\mu^y}{y!}$


---

Rewrite

$f(y_i|\mu_i) =  \frac{\mu_i^{y_i}}{y_i!}  e^{-\mu_i}$

$f(y_i|\mu_i) =  e^{ln( \frac{\mu_i^{y_i}}{y_i!} )}  e^{-\mu_i}$

$f(y_i|\mu_i) =  e^{ln(\mu_i^{y_i}) - ln( {y_i!} )}  e^{-\mu_i}$

$f(y_i|\mu_i) =  e^{-\mu_i + y_i ln(\mu_i) - ln( {y_i!} )} $ 🐵

---

Compare terms in the numerator with EDF

`Natural Parameter`: $y_i \theta_i = y_i ln(\mu_i) \implies$ $\theta_i = ln(\mu_i)$

`Cumulant Function`: $b(\theta_i) = -\mu_i \implies$ $b(\theta_i) = e^{\theta_i}$

`Dispersion Parameter`: $a(\phi) = 1$

`??`: $c(y_i) = -ln(y_i!)$ 🐵

## Binomial

$y_i \sim Bin(\pi_i, n_i)$

$f(y_i|\pi_i, n_i) = \binom{n_i}{n_iy_i} [\pi]^{n_iy_i} [1-\pi_i]^{n_i - n_iy_i}$ 


---

Rewrite

$f(y_i|\mu_i) =  \frac{\mu_i^{y_i}}{y_i!}  e^{-\mu_i}$

$f(y_i|\mu_i) =  e^{ln( \frac{\mu_i^{y_i}}{y_i!} )}  e^{-\mu_i}$

$f(y_i|\mu_i) =  e^{ln(\mu_i^{y_i}) - ln( {y_i!} )}  e^{-\mu_i}$

$f(y_i|\mu_i) =  e^{-\mu_i + y_i ln(\mu_i) - ln( {y_i!} )} $ 🐵

---

Compare terms in the numerator with EDF

`Natural Parameter`: $y_i \theta_i = y_i ln(\mu_i) \implies$ $\theta_i = ln(\mu_i)$

`Cumulant Function`: $b(\theta_i) = -\mu_i \implies$ $b(\theta_i) = e^{\theta_i}$

`Dispersion Parameter`: $a(\phi) = 1$

`??`: $c(y_i) = -ln(y_i!)$ 🐵

## Gamma

## Beta

## Chi-Squared

## Normal Distribution

# Linear Predictor

# Link Functions

# Expectations and Variances of EDFs

> Likelihood

While probability predicts outcomes from a model, the likelihood measures how well a model fits observed data. Since each data point is assumed to be a independent random observation, we can get the joint likelihood of observing all of these points (which determines the models learned probability function) by multiplying the individual probabilities for each datapoint.

<div align="center">

$$
\mathcal{l} = Pr(y_1, ... , y_n ) = Pr(y_1) Pr(y_2) Pr(y_3) ...  Pr(y_n) = \Pi_{i=1}^n f(y_i | \phi_i, \theta_i)
$$

</div>

---


> Log likelihood 

For numerical stability, we take the log of this function which turns the product into sums. This transformation is ideal because the logarithm is a `monotonically increasing function`, ensuring that maximizing $log(\mathcal{l})$ is analogous to maximizing $\mathcal{l}$ when we "undo the transformation". 

<div align="center">

$$
\mathcal{L} = log(\mathcal{l}) = \sum_{i=1}^n f(y_i | \phi_i, \theta_i)
$$

</div>

---


> Property 1: The expectation of the score function is 0. 


Wow that's a mouthful, so let's break it down. The `score function` is the first derivative (gradient) of the likelihood, with respect to the natural parameter. While we do not know in general if the likelihood function is continuous, we assume it is for computational purposes OR choose a PDF which is continuous. Our goal is to `maximize the likelihood`, which can be done by setting the derivative equal to 0 $\frac{d \mathcal{L}}{d \theta} = 0$. 

ASIDE: In order to be a <font color='red'>maximum</font>, this function must be <font color='red'>concave</font>, so how do we know the log-likelihood is concave? $\implies$ PDFs are neither strictly convex nor strictly concave, but since the log transformation turns <font color='red'>products into sums</font>, we know that each new data point will INCREASE towards a bound of 0. This is called `log-concave`

<div align="center">

$$
\frac{d}{d \theta}  \mathcal{L} = 0 \implies \frac{d}{d \theta} log(f(y_i | \phi_i, \theta_i))= 0 \implies \frac{1}{f(y_i | \phi_i, \theta_i) } f'(y_i | \phi_i, \theta_i) = 0
$$

</div>

While we can't solve for this in general, we can solve for the expecation: 

<div align="center">

$$
\mathbb{E} [\frac{1}{f(y_i | \phi_i, \theta_i) } f'(y_i | \phi_i, \theta_i)] = 0 \implies \int x \frac{1}{f(y_i | \phi_i, \theta_i) } f'(y_i | \phi_i, \theta_i) dx = 0
$$

</div>


<div align="center">

$$
\mathbb{E}[\frac{d \mathcal{L}}{d \theta}] = 0
$$

</div>


---


> Property 2: The negative Hessian equals the variance


<div align="center">

$$
-\mathbb{E}[\frac{d^2 \mathcal{L}}{d \theta^2}] = \mathbb{E}[(\frac{d \mathcal{L}}{d \theta})^2]
$$

</div>


--- 


> Connections


Using the above two facts, we get: 


<div align="center">

$$
\mathbb{E}[Y_i] = b' (\theta_i)
$$

$$
Var[Y_i] = a(\phi) b'' (\theta_i)
$$
</div>


Isn't this beautiful, it turns out that once we have the EDF form of our PDF, we can easily extract the mean (derivative of the cumulant function) and variance (second derivative times dispersion) just by looking at the natural parameter and dispersion parameter. This shouldn't be too surprising from previous intuition. For example, take the Normal distribution (above), where the natural parameter is $\mu$ and the dispersion parameter is $\sigma$... those are the only sufficient statistics that we need!



# Mean-Variance Relationship

# The likelihood equations

> Dependencies 

Our goal is to pick the parameters $\beta$ that will maximize the likelihood. However, we do so many transformations to get from $\beta$ to $\mathcal{L}$ that we can get lost in the sauce. Remember $\mathcal{L}(\theta)$, $\theta(\mu)$, $\mu(\eta)$ and $\eta(\beta)$ $\implies \mathcal{L}(\theta(\mu(\eta(\beta))))$ (so we need the chain rule for this nested model)

<div align="center">

$$
\frac{d \mathcal{L}}{d\beta} = \frac{d \mathcal{L}}{d\theta} \frac{d \theta}{d\mu} \frac{d \mu}{d\eta} \frac{d \eta}{d\beta}
$$

</div>

Earlier, we saw that
* $\frac{d \mathcal{L}}{d\theta}= \frac{y_i - b'(\theta_i)}{a(\phi)}$
* $\frac{d \theta}{d\mu}= \frac{a(\phi)}{Var(y_i)}$
* $\frac{d \eta}{d\beta}= X_{ij}$


So we get the following relationship

<div align="center">

$$
\max_{\beta} \frac{d \mathcal{L}}{d\beta} = \max_{\beta} [ (\frac{y_i - b'(\theta_i)}{a(\phi)}) (\frac{a(\phi)}{Var(y_i)}) (\frac{d \mu}{d\eta}) (X_{ij}) ]
$$

$$
0 = \frac{X_{ij} (y_i - b'(\theta_i))}{Var(y_i)} \frac{d \mu}{d\eta}
$$

</div>


---

> Connections

In matrix form, this becomes: 

<div align="center">

$$
0 = X^T D V^{-1} (Y - \mu)
$$

</div>

This should look extremely similar (actually identical) to the OLS solution

<div align="center">

$$
0 = (X^TX)^{-1} X^TY
$$

</div>

# Further References

1. [Huggingface:](https://huggingface.co/blog/charchits7/understanding-ndcgk-metric) Understnading NDCG@K
2. [Evidently AI:](https://www.evidentlyai.com/ranking-metrics/ndcg-metric) NDCG@K Explained
3. [DigitalSreeni](https://www.youtube.com/watch?v=IMvunY3LrQI&t=406s) NDCG